# Pré-processamento dos Dados — Sistema de Recomendação de Filmes

Limpeza e preparação dos dados brutos para treino do modelo. Os arquivos tratados são salvos em `data/processed/`.

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR  = Path("../data")
PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(exist_ok=True)

SAMPLE_FRAC = 0.10  # 10 % dos ratings para prototipagem
MIN_RATINGS  = 1    # filmes com menos de MIN_RATINGS avaliações são removidos

## 1. Carregamento dos dados brutos

In [2]:
ratings = pd.read_csv(RAW_DIR / "ratings.csv")
movies  = pd.read_csv(RAW_DIR / "movies.csv")

print(f"Ratings brutos : {len(ratings):,} linhas")
print(f"Filmes brutos  : {len(movies):,} linhas")

Ratings brutos : 25,000,095 linhas
Filmes brutos  : 62,423 linhas


## 2. Limpeza dos ratings

In [3]:
# Remove duplicatas exatas (mesmo usuário, mesmo filme, mesma nota)
antes = len(ratings)
ratings = ratings.drop_duplicates(subset=["userId", "movieId", "rating"])
print(f"Duplicatas removidas: {antes - len(ratings):,}")

# Converte timestamp Unix para datetime
ratings["date"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings = ratings.drop(columns=["timestamp"])

ratings.head()

Duplicatas removidas: 0


,userId,movieId,rating,date
0,1,296,5.0,2006-05-17 15:34:04
1,1,306,3.5,2006-05-17 12:26:57
2,1,307,5.0,2006-05-17 12:27:08
3,1,665,5.0,2006-05-17 15:13:40
4,1,899,3.5,2006-05-17 12:21:50


## 3. Filtragem de filmes sem avaliações

In [4]:
# Filmes que aparecem em pelo menos MIN_RATINGS avaliações
filmes_avaliados = ratings["movieId"].unique()
movies_clean = movies[movies["movieId"].isin(filmes_avaliados)].copy()

removidos = len(movies) - len(movies_clean)
print(f"Filmes sem avaliação removidos : {removidos:,}")
print(f"Filmes restantes               : {len(movies_clean):,}")

Filmes sem avaliação removidos : 3,376
Filmes restantes               : 59,047


## 4. Amostra para prototipagem rápida

In [5]:
# Amostra estratificada: mantém proporção de avaliações por usuário
ratings_sample = (
    ratings
    .groupby("userId", group_keys=False)
    .apply(lambda g: g.sample(frac=SAMPLE_FRAC, random_state=42))
    .reset_index(drop=True)
)

print(f"Ratings originais : {len(ratings):,}")
print(f"Ratings na amostra: {len(ratings_sample):,} ({SAMPLE_FRAC:.0%})")

Ratings originais : 25,000,095
Ratings na amostra: 2,497,676 (10%)


## 5. Salvamento dos arquivos tratados

In [6]:
ratings.to_csv(PROC_DIR / "ratings_clean.csv", index=False)
movies_clean.to_csv(PROC_DIR / "movies_clean.csv", index=False)
ratings_sample.to_csv(PROC_DIR / "ratings_sample.csv", index=False)

print("Arquivos salvos em data/processed/:")
for f in sorted(PROC_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name:<25} {size_mb:6.1f} MB")

Arquivos salvos em data/processed/:
  movies_clean.csv             2.7 MB
  ratings_clean.csv          843.2 MB
  ratings_sample.csv          69.1 MB


## Resumo

| Arquivo | Descrição |
|---|---|
| `ratings_clean.csv` | Todos os ratings sem duplicatas, com coluna `date` |
| `movies_clean.csv` | Apenas filmes com pelo menos 1 avaliação |
| `ratings_sample.csv` | 10% dos ratings por usuário — para prototipagem rápida |